# Bronze

## Configuración

In [ ]:
import csv
import gzip
from datetime import UTC, date, datetime
from pathlib import Path

import duckdb

def find_project_root():
    current_dir = Path.cwd()
    for candidate in (current_dir, *current_dir.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError('Open this notebook from within the project directory.')


PROJECT_ROOT = find_project_root()
DUCKDB_PATH = PROJECT_ROOT / 'data/processed/database.duckdb'
TRIPS_DIR = PROJECT_ROOT / 'data/mitma/por-distritos/viajes/diarios'

TRIPS_START_DATE = None
TRIPS_END_DATE = None

## MITMA · Viajes por distrito

In [ ]:
TRIP_COLUMNS = [
    'fecha', 'periodo', 'origen', 'destino', 'distancia',
    'actividad_origen', 'actividad_destino',
    'estudio_origen_posible', 'estudio_destino_posible',
    'residencia', 'renta', 'edad', 'sexo', 'viajes', 'viajes_km',
]

trip_files = []
for path in sorted(TRIPS_DIR.glob('????????_Viajes_distritos.csv.gz')):
    file_date = date.fromisoformat(path.name[:8])
    if TRIPS_START_DATE and file_date < TRIPS_START_DATE:
        continue
    if TRIPS_END_DATE and file_date > TRIPS_END_DATE:
        continue
    trip_files.append(path)

In [ ]:
trip_column_types = dict.fromkeys(TRIP_COLUMNS, 'VARCHAR')
trip_column_definitions = ', '.join(f'"{column}" VARCHAR' for column in TRIP_COLUMNS)
batch_timestamp = datetime.now(UTC).isoformat()

DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)
with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.execute('BEGIN TRANSACTION')
    try:
        con.execute('CREATE SCHEMA IF NOT EXISTS bronze')
        con.execute(f'''
            CREATE TABLE IF NOT EXISTS bronze.mitma_viajes (
                {trip_column_definitions},
                _archivo VARCHAR NOT NULL,
                _timestamp VARCHAR NOT NULL
            )
        ''')

        for path in trip_files:
            relative_path = path.relative_to(PROJECT_ROOT).as_posix()
            print(f'Loading {relative_path}')
            con.execute('DELETE FROM bronze.mitma_viajes WHERE _archivo = ?', [relative_path])
            con.execute('''
                INSERT INTO bronze.mitma_viajes BY NAME
                SELECT *, ? AS _archivo, ? AS _timestamp
                FROM read_csv(
                    ?, delim = '|', header = true, auto_detect = false,
                    columns = ?, nullstr = '', strict_mode = true
                )
            ''', [relative_path, batch_timestamp, str(path), trip_column_types])

        con.execute('COMMIT')
    except BaseException:
        con.execute('ROLLBACK')
        raise

print(f'Batch committed: {batch_timestamp}')

## MITMA · Pernoctaciones por distrito

In [ ]:
OVERNIGHTS_DIR = PROJECT_ROOT / 'data/mitma/por-distritos/pernoctaciones/diarios'
OVERNIGHTS_START_DATE = None
OVERNIGHTS_END_DATE = None
OVERNIGHT_COLUMNS = ['fecha', 'zona_residencia', 'zona_pernoctacion', 'personas']

overnight_files = []
for path in sorted(OVERNIGHTS_DIR.glob('????????_Pernoctaciones_distritos.csv.gz')):
    file_date = date.fromisoformat(path.name[:8])
    if OVERNIGHTS_START_DATE and file_date < OVERNIGHTS_START_DATE:
        continue
    if OVERNIGHTS_END_DATE and file_date > OVERNIGHTS_END_DATE:
        continue
    overnight_files.append(path)

In [ ]:
overnight_column_types = dict.fromkeys(OVERNIGHT_COLUMNS, 'VARCHAR')
overnight_batch_timestamp = datetime.now(UTC).isoformat()

DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)
with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.begin()
    con.execute('CREATE SCHEMA IF NOT EXISTS bronze')
    con.execute('''
        CREATE TABLE IF NOT EXISTS bronze.mitma_pernoctaciones (
            fecha VARCHAR,
            zona_residencia VARCHAR,
            zona_pernoctacion VARCHAR,
            personas VARCHAR,
            _archivo VARCHAR NOT NULL,
            _timestamp VARCHAR NOT NULL
        )
    ''')

    for path in overnight_files:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path}')
        con.execute('DELETE FROM bronze.mitma_pernoctaciones WHERE _archivo = ?', [relative_path])
        con.execute('''
            INSERT INTO bronze.mitma_pernoctaciones BY NAME
            SELECT *, ? AS _archivo, ? AS _timestamp
            FROM read_csv(
                ?, delim = '|', header = true, auto_detect = false,
                columns = ?, nullstr = '', strict_mode = true
            )
        ''', [relative_path, overnight_batch_timestamp, str(path), overnight_column_types])

    con.commit()

print(f'Batch committed: {overnight_batch_timestamp}')

## MITMA · Zonificación

In [ ]:
ZONING_DIR = PROJECT_ROOT / 'data/mitma/zonificación'
DISTRICTS_DIR = ZONING_DIR / 'zonificacion_distritos'
ZONING_SOURCES = [
    ('bronze.mitma_zonificacion_distritos', DISTRICTS_DIR / 'zonificacion_distritos.shp'),
    ('bronze.mitma_centroides_distritos', DISTRICTS_DIR / 'zonificacion_distritos_centroides.shp'),
    ('bronze.mitma_relacion_ine', ZONING_DIR / 'relacion_ine_zonificacionMitma.csv'),
]
ZONE_NAMES_TABLE = 'bronze.mitma_nombres_zonas'
ZONE_NAME_SOURCES = [
    ('distrito', DISTRICTS_DIR / 'nombres_distritos.csv'),
    ('municipio', ZONING_DIR / 'zonificacion_municipios/nombres_municipios.csv'),
    ('gau', ZONING_DIR / 'zonificacion_GAU/nombres_gaus.csv'),
]


zoning_batch_timestamp = datetime.now(UTC).isoformat()
DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)

with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.execute('INSTALL spatial')
    con.execute('LOAD spatial')
    con.begin()
    con.execute('CREATE SCHEMA IF NOT EXISTS bronze')

    for table, path in ZONING_SOURCES:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path} into {table}')
        if path.suffix == '.shp':
            source_sql = '''
                SELECT ID, ST_AsText(geom) AS geom_wkt, ST_CRS(geom) AS crs
                FROM ST_Read(?)
            '''
        else:
            source_sql = '''
                SELECT * FROM read_csv(
                    ?, delim = '|', header = true, all_varchar = true,
                    nullstr = '', strict_mode = true
                )
            '''
        con.execute(f'''
            CREATE OR REPLACE TABLE {table} AS
            SELECT *, ? AS _archivo, ? AS _timestamp
            FROM ({source_sql}) AS source
        ''', [relative_path, zoning_batch_timestamp, str(path)])

    con.execute(f'''
        CREATE OR REPLACE TABLE {ZONE_NAMES_TABLE} (
            ID VARCHAR,
            name VARCHAR,
            tipo_zona VARCHAR NOT NULL,
            _archivo VARCHAR NOT NULL,
            _timestamp VARCHAR NOT NULL
        )
    ''')
    for zone_type, path in ZONE_NAME_SOURCES:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path} into {ZONE_NAMES_TABLE}')
        con.execute(f'''
            INSERT INTO {ZONE_NAMES_TABLE} BY NAME
            SELECT ID, name, ? AS tipo_zona, ? AS _archivo, ? AS _timestamp
            FROM read_csv(
                ?, delim = '|', header = true, all_varchar = true,
                nullstr = '', strict_mode = true
            )
        ''', [zone_type, relative_path, zoning_batch_timestamp, str(path)])

    con.commit()

print(f'Batch committed: {zoning_batch_timestamp}')

## INE · Población, estudios y situación laboral

In [ ]:
INE_DIR = PROJECT_ROOT / 'data/ine'
INE_SOURCES = [
    ('bronze.ine_poblacion', INE_DIR / 'poblacion.csv', '\t'),
    ('bronze.ine_nivel_estudios', INE_DIR / 'nivel_estudios.csv', '\t'),
    ('bronze.ine_relacion_actividad', INE_DIR / 'relacion_actividad.csv', '\t'),
]


ine_batch_timestamp = datetime.now(UTC).isoformat()
DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)

with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.begin()
    con.execute('CREATE SCHEMA IF NOT EXISTS bronze')
    for table, path, delimiter in INE_SOURCES:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path} into {table}')
        con.execute(f'''
            CREATE OR REPLACE TABLE {table} AS
            SELECT *, ? AS _archivo, ? AS _timestamp
            FROM read_csv(
                ?, delim = ?, header = true, all_varchar = true,
                nullstr = '', null_padding = true, strict_mode = true
            )
        ''', [relative_path, ine_batch_timestamp, str(path), delimiter])
    con.commit()

print(f'Batch committed: {ine_batch_timestamp}')

## INE · Catálogos territoriales

In [ ]:
INE_CATALOG_DIR = PROJECT_ROOT / 'data/ine'
INE_CATALOG_SOURCES = [
    ('bronze.ine_municipios', INE_CATALOG_DIR / 'diccionario26.xlsx', '''
        read_xlsx(?, range = 'A2:E1048576', header = true,
                  all_varchar = true, stop_at_empty = true)
    '''),
    ('bronze.ine_provincias_ccaa', INE_CATALOG_DIR / 'provincias_ccaa.csv', '''
        read_csv(?, delim = ';', header = true, all_varchar = true,
                 nullstr = '', strict_mode = true)
    '''),
]


catalog_batch_timestamp = datetime.now(UTC).isoformat()
DUCKDB_PATH.parent.mkdir(parents=True, exist_ok=True)

with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.execute('INSTALL excel')
    con.execute('LOAD excel')
    con.begin()
    con.execute('CREATE SCHEMA IF NOT EXISTS bronze')
    for table, path, reader in INE_CATALOG_SOURCES:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path} into {table}')
        con.execute(f'''
            CREATE OR REPLACE TABLE {table} AS
            SELECT *, ? AS _archivo, ? AS _timestamp
            FROM {reader}
        ''', [relative_path, catalog_batch_timestamp, str(path)])
    con.commit()

print(f'Batch committed: {catalog_batch_timestamp}')

## INE · Renta y rama de actividad

In [ ]:
INE_PREFIX = ['Total Nacional', 'Provincias', 'Municipios', 'Secciones', 'Sexo']
INE_RENTA_ACTIVIDAD_SOURCES = [
    ('bronze.ine_renta', INE_DIR / 'renta.csv', 'utf-8',
     ['Municipios', 'Distritos', 'Secciones', 'Indicadores de renta media', 'Periodo', 'Total']),
    ('bronze.ine_rama_actividad', INE_DIR / 'rama_actividad.csv', 'utf-8',
     INE_PREFIX + ['Actividad económica', 'Periodo', 'Total']),
]

ine_renta_actividad_timestamp = datetime.now(UTC).isoformat()
with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.begin()
    con.execute('CREATE SCHEMA IF NOT EXISTS bronze')
    for table, path, encoding, columns in INE_RENTA_ACTIVIDAD_SOURCES:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path} into {table}')
        con.execute(f'''
            CREATE OR REPLACE TABLE {table} AS
            SELECT *, ? AS _archivo, ? AS _timestamp
            FROM read_csv(
                ?, delim = ?, encoding = ?, header = true, auto_detect = false,
                columns = ?, nullstr = '', strict_mode = true
            )
        ''', [relative_path, ine_renta_actividad_timestamp, str(path), '\t', encoding, dict.fromkeys(columns, 'VARCHAR')])
    con.commit()

print(f'Batch committed: {ine_renta_actividad_timestamp}')

## DGT · Datos municipales

In [ ]:
DGT_TABLE = 'bronze.dgt_datos_municipales'
DGT_YEARS = [2022, 2023, 2024]
DGT_SOURCES = [PROJECT_ROOT / f'data/dgt/DatosMunicipalesGeneral_{year}.xlsx' for year in DGT_YEARS]
DGT_READER = 'read_xlsx(?, header = true, all_varchar = true)'


dgt_batch_timestamp = datetime.now(UTC).isoformat()
with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.execute('INSTALL excel')
    con.execute('LOAD excel')
    con.begin()
    con.execute('CREATE SCHEMA IF NOT EXISTS bronze')
    con.execute(f'''
        CREATE TABLE IF NOT EXISTS {DGT_TABLE} AS
        SELECT *, '' AS _archivo, '' AS _timestamp FROM {DGT_READER} LIMIT 0
    ''', [str(DGT_SOURCES[0])])
    for path in DGT_SOURCES:
        relative_path = path.relative_to(PROJECT_ROOT).as_posix()
        print(f'Loading {relative_path} into {DGT_TABLE}')
        con.execute(f'DELETE FROM {DGT_TABLE} WHERE _archivo = ?', [relative_path])
        con.execute(f'''
            INSERT INTO {DGT_TABLE} BY NAME
            SELECT *, ? AS _archivo, ? AS _timestamp FROM {DGT_READER}
        ''', [relative_path, dgt_batch_timestamp, str(path)])
    con.commit()

print(f'Batch committed: {dgt_batch_timestamp}')

## BOE · Calendario laboral

In [ ]:
BOE_TABLE = 'bronze.boe_calendario_festivos_2023'
BOE_PATH = PROJECT_ROOT / 'data/boe/calendario_festivos_2023.csv'
BOE_COLUMNS = [
    'Fecha de las fiestas', 'Andalucía', 'Aragón', 'Asturias', 'Illes Balears', 'Canarias (1)',
    'Cantabria', 'Castilla-La Mancha', 'Castilla y León', 'Cataluña (2)', 'Extremadura', 'Galicia',
    'Com. Madrid', 'Región Murcia', 'C. Foral Navarra', 'País Vasco', 'La Rioja',
    'Comunitat Valenciana', 'Ciudad de Ceuta (3)', 'Ciudad de Melilla',
]

boe_batch_timestamp = datetime.now(UTC).isoformat()
relative_path = BOE_PATH.relative_to(PROJECT_ROOT).as_posix()
with duckdb.connect(str(DUCKDB_PATH)) as con:
    con.begin()
    con.execute('CREATE SCHEMA IF NOT EXISTS bronze')
    print(f'Loading {relative_path} into {BOE_TABLE}')
    con.execute(f'''
        CREATE OR REPLACE TABLE {BOE_TABLE} AS
        SELECT *, ? AS _archivo, ? AS _timestamp
        FROM read_csv(
            ?, delim = ';', header = true, auto_detect = false,
            columns = ?, nullstr = '', strict_mode = true
        )
    ''', [relative_path, boe_batch_timestamp, str(BOE_PATH), dict.fromkeys(BOE_COLUMNS, 'VARCHAR')])
    con.commit()

print(f'Batch committed: {boe_batch_timestamp}')

## Copernicus · Series horarias

In [ ]:
import sys
import pandas as pd

sys.path.insert(0, str(PROJECT_ROOT))
from scripts.download_copernicus import OUTPUT, VARIABLES, available_assets, read_hourly_asset

with duckdb.connect(str(DUCKDB_PATH)) as con:
    fields = ', '.join(f'"{k}" VARCHAR' for k in VARIABLES)
    con.execute(f"""CREATE TABLE IF NOT EXISTS bronze.copernicus_horario
        (id_distrito VARCHAR, hora_utc VARCHAR, {fields}, _archivo VARCHAR, _timestamp VARCHAR)""")
    for folder in sorted(OUTPUT.glob('*/')):
        for period, path in available_assets(OUTPUT, folder.name):
            weather, _ = read_hourly_asset(path, period)
            batch = weather.assign(id_distrito=folder.name, hora_utc=weather.index.strftime('%Y-%m-%dT%H:%M:%SZ'),
                                   _archivo=path.relative_to(PROJECT_ROOT).as_posix(), _timestamp=datetime.now(UTC).isoformat())
            con.begin()
            con.execute('DELETE FROM bronze.copernicus_horario WHERE id_distrito = ? AND hora_utc BETWEEN ? AND ?',
                        [folder.name, batch.hora_utc.iloc[0], batch.hora_utc.iloc[-1]])
            con.execute('INSERT INTO bronze.copernicus_horario BY NAME SELECT * FROM batch')
            con.commit()